# 6.17 — Residual & Skip Connections

Residual and skip connections let a deep model carry an activation forward unchanged while a learned branch adds only a correction. In this lesson, we build the residual map `y = x + F(x)` from scratch, inspect why its derivative contains an identity path, and use NumPy to see how shortcuts stabilize signals, gradients, shapes, and practical bookkeeping.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build residual and skip connections one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is exposed so the shortcut never feels like magic. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, matrix products, Jacobians, and numerical checks.
import matplotlib.pyplot as plt  # small visualizations for signals, gradients, and training curves.
np.random.seed(0)  # reproducibility for the tiny training demonstrations.

### 1. The residual map: learn a correction, not a replacement

A plain layer computes a new representation from scratch: `y = F(x)`. A residual block instead computes `y = x + F(x)`. The shortcut `x` says, "if the learned branch is not useful yet, at least pass the current signal through." The residual branch `F(x)` only has to learn a correction to the identity path, which is often easier than relearning the whole transformation.

In [ ]:
x_w = np.array([1.5, -0.5, 2.0])  # incoming activation vector from a previous layer.
F_w = np.array([0.2, 0.1, -0.4])  # learned correction produced by a residual branch.
y_plain_w = F_w  # a plain block would replace x with only the branch output.
y_res_w = x_w + F_w  # a residual block keeps x and adds the learned correction.
print("x:", x_w)  # inspect the carried signal.
print("F(x):", F_w)  # inspect the learned correction.
print("plain y=F(x):", y_plain_w)  # replacement.
print("residual y=x+F(x):", y_res_w)  # correction around the identity.
assert np.allclose(y_res_w, [1.7, -0.4, 1.6])  # concrete residual sum.

▶ What you'll see: the residual output stays close to the input because the branch adds a small correction.

In [ ]:
plt.figure(figsize=(5, 3))  # create a compact comparison chart.
idx_w = np.arange(len(x_w))  # one bar group per coordinate.
plt.bar(idx_w - 0.22, x_w, width=0.22, label="x", color="gray")  # show the shortcut signal.
plt.bar(idx_w, F_w, width=0.22, label="F(x)", color="orange")  # show the correction.
plt.bar(idx_w + 0.22, y_res_w, width=0.22, label="x+F(x)", color="teal")  # show the residual output.
plt.axhline(0, color="black", linewidth=0.7)  # separate positive and negative activations.
plt.title("1: residual output = shortcut + correction")  # title the figure.
plt.xlabel("coordinate")  # label activation coordinate.
plt.ylabel("value")  # label activation magnitude.
plt.legend()  # show which bars are which.
plt.show()  # display the plot.

▶ What you'll see: the teal bars are the gray shortcut bars nudged by the orange correction bars.

*Why it's done this way:* deep layers are composed many times, so forcing every layer to rewrite the whole representation makes optimization fragile. The identity path makes the default behavior safe: `F(x)=0` gives `y=x`. Learning then becomes a local correction problem, which is mathematically easier because the branch only needs to model the difference between the desired output and the current signal.

### 2. A residual branch can be an ordinary affine-and-ReLU computation

The residual branch does not need special machinery. It can be a small neural computation: affine signal, nonlinearity, then an output vector. The lesson block uses a two-input scratch pass with `x=[1.5,-0.5]`, weights `[1.4,-0.4]`, and bias `0.8`, giving an affine value of `3.1` before ReLU.

In [ ]:
x2_w = np.array([1.5, -0.5])  # two-input activation from the lesson arithmetic.
w2_w = np.array([1.4, -0.4])  # branch weights.
b0_w = 0.8  # branch bias.
affine_w = float(w2_w @ x2_w + b0_w)  # 1.4*1.5 + (-0.4)*(-0.5) + 0.8.
gated_w = max(0.0, affine_w)  # ReLU gate keeps positive signals and clips negative ones.
print("affine:", round(affine_w, 3))  # inspect the pre-activation.
print("ReLU(affine):", round(gated_w, 3))  # inspect the gated branch signal.
assert round(affine_w, 3) == 3.100  # lesson number.
assert round(gated_w, 3) == 3.100  # positive signal passes through ReLU.

▶ What you'll see: the visible arithmetic produces `3.100`, and ReLU leaves it unchanged because it is positive.

In [ ]:
v_out_w = np.array([0.3, -0.2])  # map the scalar hidden signal back to the input shape.
F2_w = gated_w * v_out_w  # residual branch output has the same length as x2.
y2_w = x2_w + F2_w  # add shortcut and branch elementwise.
print("F(x):", np.round(F2_w, 3))  # inspect learned correction vector.
print("y=x+F(x):", np.round(y2_w, 3))  # inspect residual output.
assert np.allclose(np.round(F2_w, 3), [0.93, -0.62])  # 3.1*[0.3,-0.2].

▶ What you'll see: the scalar branch signal becomes a same-shape correction before addition.

In [ ]:
plt.figure(figsize=(5, 3))  # create a small signal-flow chart.
plt.bar(["affine", "ReLU", "F0", "F1"], [affine_w, gated_w, F2_w[0], F2_w[1]], color=["gray", "seagreen", "orange", "orange"])  # show branch stages.
plt.axhline(0, color="black", linewidth=0.7)  # zero reference for ReLU and signed correction.
plt.title("2: inside the residual branch")  # title the plot.
plt.ylabel("value")  # label the signal value.
plt.show()  # display the plot.

▶ What you'll see: the branch first creates one positive hidden signal, then turns it into positive and negative corrections.

*Why it's done this way:* the shortcut addition is simple only when the branch output has the same shape as the input. The affine-and-ReLU branch can be arbitrarily expressive, but the final vector must align coordinate-by-coordinate with `x` so `x + F(x)` is a valid elementwise map.

### 3. The identity derivative gives gradients a clean shortcut

For `y = x + F(x)`, the derivative with respect to `x` is `I + dF/dx`. The `I` term is the shortcut gradient: even if the residual branch has a tiny derivative, the backward pass still has a direct route through the identity. This is the central optimization reason residual networks train deeply.

In [ ]:
J_F_w = np.array([[0.10, 0.00, 0.02],  # a small residual-branch Jacobian.
                  [0.00, -0.05, 0.01],
                  [0.03, 0.00, 0.08]])
I_w = np.eye(3)  # derivative of x with respect to x.
J_res_w = I_w + J_F_w  # derivative of x + F(x).
print("dF/dx:\n", J_F_w)  # inspect branch derivative.
print("I + dF/dx:\n", J_res_w)  # inspect residual derivative.
assert np.allclose(np.diag(J_res_w), [1.10, 0.95, 1.08])  # identity shifts diagonal near 1.

▶ What you'll see: the residual Jacobian has diagonal entries near one, not near zero.

In [ ]:
grad_out_w = np.array([1.0, -0.5, 0.2])  # gradient arriving from later layers.
grad_plain_w = J_F_w.T @ grad_out_w  # gradient through the branch alone.
grad_res_w = J_res_w.T @ grad_out_w  # gradient through shortcut plus branch.
print("plain branch gradient:", np.round(grad_plain_w, 3))  # small because J_F is small.
print("residual gradient:", np.round(grad_res_w, 3))  # includes the identity route.
assert np.linalg.norm(grad_res_w) > np.linalg.norm(grad_plain_w)  # shortcut preserves signal.

▶ What you'll see: the residual gradient remains close to the incoming gradient, while the branch-only gradient is much smaller.

In [ ]:
plt.figure(figsize=(5, 3))  # create a compact norm comparison.
plt.bar(["incoming", "branch only", "residual"], [np.linalg.norm(grad_out_w), np.linalg.norm(grad_plain_w), np.linalg.norm(grad_res_w)], color=["gray", "red", "teal"])  # compare gradient magnitudes.
plt.title("3: the identity path preserves gradient scale")  # title the plot.
plt.ylabel("gradient norm")  # label norm scale.
plt.show()  # display the plot.

▶ What you'll see: the residual bar is far closer to the incoming gradient norm than the branch-only bar.

*Why it's done this way:* backpropagation multiplies many local derivatives. If every layer contributes only a small branch Jacobian, gradients vanish quickly. Adding `I` means every residual block contributes a path whose derivative is exactly one, so the product across many blocks has a stable component even before the learned branch is well tuned.

### 4. Many residual blocks preserve signal better than many plain small-gain blocks

A deep plain stack repeatedly applies branch gains. If the typical gain is below one, activations and gradients shrink. A residual stack repeatedly applies `1 + small gain`, so the identity component prevents immediate collapse. This does not make every residual stack automatically stable, but it changes the default multiplication pattern.

In [ ]:
depths_w = np.arange(1, 31)  # compare depth from 1 to 30 blocks.
plain_gain_w = 0.8 ** depths_w  # repeated small gain without a shortcut.
res_gain_w = 1.02 ** depths_w  # identity plus small positive branch gain.
print("plain gain at depth 30:", round(float(plain_gain_w[-1]), 4))  # 0.8^30.
print("residual gain at depth 30:", round(float(res_gain_w[-1]), 4))  # 1.02^30.
assert round(float(plain_gain_w[-1]), 4) == 0.0012  # concrete vanishing number.

▶ What you'll see: multiplying by `0.8` thirty times almost erases the signal.

In [ ]:
plt.figure(figsize=(5, 3))  # create a depth comparison plot.
plt.plot(depths_w, plain_gain_w, marker="o", label="plain: 0.8^L", color="red")  # plain repeated gain.
plt.plot(depths_w, res_gain_w, marker="o", label="residual: 1.02^L", color="teal")  # residual-style gain.
plt.yscale("log")  # log scale makes the decay visible.
plt.title("4: depth multiplies local gains")  # title the plot.
plt.xlabel("number of blocks")  # label depth axis.
plt.ylabel("signal/gradient multiplier")  # label multiplier.
plt.legend()  # show curve labels.
plt.show()  # display the plot.

▶ What you'll see: the plain curve falls rapidly on a log scale, while the residual-style curve stays near order one.

*Why it's done this way:* the shortcut changes deep learning from repeatedly multiplying only `dF/dx` to repeatedly multiplying `I + dF/dx`. That does not remove the need for good initialization and normalization, but it gives the optimization a path whose scale starts near one instead of near the branch gain.

### 5. Training the branch as a correction uses small reliable nudges

If the target transformation is close to the input, the residual branch should learn `target - x`. For one scalar parameter, the lesson's update uses learning rate `η=0.070`, gradient `g=1.650`, and parameter `2.000`, giving `1.885`. The key idea is not the specific parameter; it is the small repeated correction.

In [ ]:
param_w = 2.0  # one scalar branch parameter before an update.
eta_w = 0.070  # learning rate from the lesson block.
g_w = 1.650  # gradient for this scalar parameter.
new_param_w = param_w - eta_w * g_w  # gradient descent update.
print("old parameter:", round(param_w, 3))  # inspect before.
print("step ηg:", round(eta_w * g_w, 3))  # inspect the movement size.
print("new parameter:", round(new_param_w, 3))  # inspect after.
assert round(new_param_w, 3) == 1.885  # lesson update number.

▶ What you'll see: the update moves the parameter by only `0.116`, not by a huge jump.

In [ ]:
x_train_w = np.linspace(-2, 2, 25)  # tiny one-dimensional training inputs.
target_w = x_train_w + 0.4 * np.sin(x_train_w)  # target is identity plus a smooth correction.
a_w = 0.0  # residual branch starts at zero correction: F(x)=a*x.
losses_train_w = []  # record loss during training.
for step_w in range(80):  # small gradient descent loop.
    pred_w = x_train_w + a_w * x_train_w  # residual prediction x + F(x).
    err_w = pred_w - target_w  # residual error.
    grad_a_w = 2 * np.mean(err_w * x_train_w)  # derivative of mean squared error wrt a.
    a_w -= 0.05 * grad_a_w  # small parameter update.
    losses_train_w.append(float(np.mean(err_w ** 2)))  # save loss.
print("learned correction slope:", round(a_w, 3))  # inspect learned residual parameter.
print("loss start -> end:", round(losses_train_w[0], 4), "->", round(losses_train_w[-1], 4))  # inspect convergence.
assert losses_train_w[-1] < losses_train_w[0]  # training improved the correction.

▶ What you'll see: the loss decreases because the branch learns a nonzero correction around the identity.

In [ ]:
plt.figure(figsize=(5, 3))  # create a training curve plot.
plt.plot(losses_train_w, color="purple")  # show residual training loss.
plt.title("5: residual branch learns a correction")  # title the plot.
plt.xlabel("step")  # label step axis.
plt.ylabel("mean squared error")  # label loss.
plt.show()  # display the curve.

▶ What you'll see: a smooth decreasing curve, showing that repeated small nudges can learn the correction.

*Why it's done this way:* gradient descent is trustworthy when each step is local enough for the derivative to remain a good guide. Residual learning helps because the branch models the error of the identity path; if the identity is already close, the target correction is smaller and easier to learn reliably.

### 6. Turning residual scores into comparisons with softmax

Residual blocks often feed classifiers or attention modules, where a raw score must become a comparison. The lesson compares score `3.100` with baseline `0.400`: exponentiate both and normalize. The residual idea still matters because stable activations make these downstream probabilities less erratic.

In [ ]:
score_w = 3.100  # lesson score from the residual branch scratch pass.
baseline_w = 0.400  # comparison score.
exp_score_w = np.exp(score_w)  # unnormalized positive evidence for the lesson score.
exp_base_w = np.exp(baseline_w)  # unnormalized positive evidence for the baseline.
prob_w = exp_score_w / (exp_score_w + exp_base_w)  # two-class softmax probability.
print("exp(3.1):", round(float(exp_score_w), 3))  # inspect numerator.
print("exp(0.4):", round(float(exp_base_w), 3))  # inspect competing score.
print("softmax probability:", round(float(prob_w), 3))  # normalized comparison.
assert round(float(prob_w), 3) == 0.937  # lesson softmax number.

▶ What you'll see: `3.100` becomes a high but not absolute probability of `0.937` against `0.400`.

In [ ]:
scores_grid_w = np.linspace(-1, 5, 80)  # possible lesson scores against the fixed baseline.
probs_grid_w = np.exp(scores_grid_w) / (np.exp(scores_grid_w) + exp_base_w)  # softmax probability curve.
plt.figure(figsize=(5, 3))  # create a probability curve.
plt.plot(scores_grid_w, probs_grid_w, color="teal")  # show score-to-probability mapping.
plt.scatter([score_w], [prob_w], color="red")  # mark the lesson score.
plt.axhline(0.5, color="gray", linestyle="--")  # show equal-comparison probability.
plt.title("6: softmax turns scores into comparisons")  # title the plot.
plt.xlabel("score compared with baseline 0.4")  # label score axis.
plt.ylabel("probability")  # label probability axis.
plt.show()  # display the plot.

▶ What you'll see: probability rises smoothly with score, with the lesson point high on the curve.

*Why it's done this way:* exponentials convert arbitrary real-valued scores into positive evidence, and division by the evidence sum makes a calibrated comparison. If residual blocks keep score scales controlled, the softmax sits in a useful range where gradients are not instantly saturated.

### 7. Scale, normalization, shape, and memory bookkeeping

A shortcut can only be added when shapes match, and stable scale still matters. The lesson normalizes `3.100` using mean `1.000` and variance `0.250`, giving `4.200`, and counts a tiny activation block with `3` vectors of length `128` as `1.500` KB in 32-bit floats. These numbers remind us that architecture choices are also numerical and hardware choices.

In [ ]:
value_w = 3.100  # residual signal to normalize.
mean_w = 1.000  # running or batch mean.
var_w = 0.250  # running or batch variance.
eps_w = 1e-5  # numerical stabilizer.
normalized_w = (value_w - mean_w) / np.sqrt(var_w + eps_w)  # standard normalization formula.
print("normalized value:", round(float(normalized_w), 3))  # inspect scale-adjusted signal.
assert round(float(normalized_w), 3) == 4.200  # lesson normalization number.

▶ What you'll see: the signal is `4.200` standard deviations above the mean, so it is large after normalization.

In [ ]:
vectors_w = 3  # three activation vectors in the tiny block.
length_w = 128  # each vector length.
bytes_per_float_w = 4  # float32 size.
kb_w = vectors_w * length_w * bytes_per_float_w / 1024  # memory in KB.
print("activation memory KB:", round(kb_w, 3))  # inspect memory bookkeeping.
assert round(kb_w, 3) == 1.500  # lesson memory number.

▶ What you'll see: even a tiny residual block stores activation memory; depth and batch size multiply it.

In [ ]:
x_bad_w = np.ones(3)  # shortcut with length 3.
F_bad_w = np.ones(2)  # branch output with length 2, not addable to x_bad.
W_proj_w = np.array([[1., 0., 0.], [0., 1., 0.]])  # projection that maps length 3 to length 2.
x_proj_w = W_proj_w @ x_bad_w  # projected shortcut now has length 2.
y_proj_w = x_proj_w + F_bad_w  # valid projected skip addition.
print("projected shortcut shape:", x_proj_w.shape)  # inspect shape after projection.
print("projected residual output:", y_proj_w)  # inspect valid addition result.
assert y_proj_w.shape == F_bad_w.shape  # concrete shape compatibility check.

▶ What you'll see: a projection fixes the shape mismatch so the shortcut and branch can be added.

*Why it's done this way:* normalization controls the effective scale seen by later layers and gradients, while projections preserve the algebra of addition when dimensions change. The memory calculation matters because residual networks keep activations for backpropagation, so mathematical elegance still has a hardware cost.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, matrix products, gradients, and numerical assertions.
import matplotlib.pyplot as plt  # load Matplotlib for every diagnostic plot in the lesson.
np.random.seed(0)  # make the examples reproducible across notebook runs.

def relu(z):  # define ReLU from scratch.
    return np.maximum(0.0, z)  # clip negative values to zero and leave positive values unchanged.

def residual(x, fx):  # define the residual addition rule.
    return x + fx  # add the shortcut and branch elementwise.

def softmax2(a, b):  # define a stable two-score softmax.
    m = max(float(a), float(b))  # subtract the max so exponentials stay numerically safe.
    ea = np.exp(a - m)  # exponentiate shifted first score.
    eb = np.exp(b - m)  # exponentiate shifted second score.
    return float(ea / (ea + eb))  # normalize into a probability for the first score.

def show_vec(values, labels, title):  # define a tiny vector plotting helper.
    plt.figure(figsize=(4.5, 3))  # create a compact figure.
    plt.bar(labels, values, color="teal")  # draw values as bars.
    plt.axhline(0, color="black", linewidth=0.7)  # show the zero reference.
    plt.title(title)  # add a title.
    plt.show()  # display the plot.

## 🟢 Basics (warm-up)

### Basic 1 — Add a shortcut to a correction

**Goal.** Build `y = x + F(x)` with tiny vectors, because residual blocks keep the old signal and add a learned change. We build it in 2 steps.

In [ ]:
x_b1 = np.array([1.0, 2.0, -1.0])  # define a small activation vector.
F_b1 = np.array([0.2, -0.5, 0.1])  # define a small learned correction.
print("shortcut x:", x_b1)  # inspect the carried signal.
print("correction F(x):", F_b1)  # inspect the residual branch output.

▶ What you'll see: two same-length vectors ready for elementwise addition.

In [ ]:
y_b1 = residual(x_b1, F_b1)  # add shortcut and correction.
print("residual output:", y_b1)  # inspect the corrected activation.
assert np.allclose(y_b1, [1.2, 1.5, -0.9])  # verify the coordinatewise sum.
show_vec(y_b1, ["c0", "c1", "c2"], "Basic 1: x + F(x)")  # visualize the output coordinates.

▶ What you'll see: each output coordinate is the corresponding input coordinate nudged by `F(x)`.

👀 Takeaway: a residual block learns a correction around the identity path.

### Basic 2 — Check the zero-branch identity case

**Goal.** Set `F(x)=0`, because a residual block should be able to behave like the identity when the branch has nothing useful to add. We build it in 2 steps.

In [ ]:
x_b2 = np.array([3.0, -2.0, 0.5])  # define an incoming activation.
F_b2 = np.zeros_like(x_b2)  # define a zero residual branch.
print("x:", x_b2)  # inspect the input.
print("zero branch:", F_b2)  # inspect the branch output.

▶ What you'll see: the branch contributes no change.

In [ ]:
y_b2 = residual(x_b2, F_b2)  # residual output with zero correction.
print("y:", y_b2)  # inspect the output.
assert np.allclose(y_b2, x_b2)  # residual block exactly copies x.
show_vec(y_b2 - x_b2, ["c0", "c1", "c2"], "Basic 2: output minus input")  # visualize no change.

▶ What you'll see: all difference bars are zero.

👀 Takeaway: the identity path is the safe default behavior of a residual block.

### Basic 3 — Compute the lesson affine signal

**Goal.** Reproduce the scratch affine calculation `1.4·1.5 + (-0.4)·(-0.5) + 0.8`, because residual branches are ordinary neural computations. We build it in 2 steps.

In [ ]:
x_b3 = np.array([1.5, -0.5])  # lesson input vector.
w_b3 = np.array([1.4, -0.4])  # lesson branch weights.
b_b3 = 0.8  # lesson branch bias.
products_b3 = w_b3 * x_b3  # compute coordinate contributions.
print("products:", products_b3)  # inspect 2.1 and 0.2.

▶ What you'll see: each input coordinate contributes to the affine signal.

In [ ]:
affine_b3 = float(np.sum(products_b3) + b_b3)  # add products and bias.
print("affine value:", round(affine_b3, 3))  # inspect the lesson number.
assert round(affine_b3, 3) == 3.100  # verify the concrete arithmetic.
show_vec([products_b3[0], products_b3[1], b_b3], ["w0x0", "w1x1", "bias"], "Basic 3: affine pieces")  # visualize the sum pieces.

▶ What you'll see: the pieces add to `3.100`.

👀 Takeaway: residual branches still start from simple affine arithmetic.

### Basic 4 — Apply a ReLU gate

**Goal.** Gate positive and negative branch signals with ReLU, because nonlinear residual branches reshape activations before the shortcut addition. We build it in 2 steps.

In [ ]:
z_b4 = np.array([-1.2, 0.0, 3.1])  # define pre-activation values.
print("pre-activation:", z_b4)  # inspect values before gating.

▶ What you'll see: one negative value, one zero, and one positive value.

In [ ]:
h_b4 = relu(z_b4)  # apply ReLU elementwise.
print("after ReLU:", h_b4)  # inspect gated values.
assert np.allclose(h_b4, [0.0, 0.0, 3.1])  # negative clips, positive passes.
show_vec(h_b4, ["neg", "zero", "pos"], "Basic 4: ReLU gate")  # visualize gated activations.

▶ What you'll see: negative values disappear while positive values pass through.

👀 Takeaway: the residual branch can be nonlinear even though the shortcut is an identity.

### Basic 5 — Add a scalar branch signal back to matching shape

**Goal.** Convert a hidden scalar into a correction vector, because the residual addition requires matching output shape. We build it in 2 steps.

In [ ]:
x_b5 = np.array([1.5, -0.5])  # shortcut input.
h_b5 = 3.1  # scalar hidden branch signal from the lesson pass.
v_b5 = np.array([0.3, -0.2])  # output weights mapping scalar to two coordinates.
F_b5 = h_b5 * v_b5  # same-shape correction vector.
print("F(x):", np.round(F_b5, 3))  # inspect correction.

▶ What you'll see: the branch produces a two-coordinate correction.

In [ ]:
y_b5 = x_b5 + F_b5  # add shortcut and branch.
print("residual output:", np.round(y_b5, 3))  # inspect corrected vector.
assert np.allclose(np.round(y_b5, 3), [2.43, -1.12])  # verify addition.
show_vec(y_b5, ["c0", "c1"], "Basic 5: same-shape residual output")  # visualize the two output coordinates.

▶ What you'll see: coordinate 0 increases and coordinate 1 decreases after the correction.

👀 Takeaway: shape compatibility is a mathematical requirement, not a software detail.

### Basic 6 — Inspect the residual Jacobian diagonal

**Goal.** Build `I + dF/dx`, because the shortcut derivative is what gives residual blocks their clean gradient path. We build it in 2 steps.

In [ ]:
J_F_b6 = np.array([[0.1, 0.0], [0.0, -0.2]])  # branch derivative for a two-coordinate block.
I_b6 = np.eye(2)  # identity derivative from the shortcut.
print("branch Jacobian:\n", J_F_b6)  # inspect learned derivative.

▶ What you'll see: the branch derivative is small compared with an identity matrix.

In [ ]:
J_res_b6 = I_b6 + J_F_b6  # residual derivative.
print("residual Jacobian:\n", J_res_b6)  # inspect I + dF/dx.
assert np.allclose(np.diag(J_res_b6), [1.1, 0.8])  # verify identity-shifted diagonal.
plt.figure(figsize=(4, 3))  # create a heatmap.
plt.imshow(J_res_b6, cmap="coolwarm", vmin=-1, vmax=1)  # visualize derivative entries.
plt.colorbar(label="derivative")  # add numeric scale.
plt.title("Basic 6: I + dF/dx")  # title the heatmap.
plt.show()  # display the plot.

▶ What you'll see: the diagonal stays near one because of the shortcut.

👀 Takeaway: residual derivatives include an identity term that helps gradients flow backward.

### Basic 7 — Compare one backward gradient with and without a shortcut

**Goal.** Multiply an incoming gradient through a branch-only Jacobian and a residual Jacobian, because the difference shows the identity path numerically. We build it in 2 steps.

In [ ]:
grad_out_b7 = np.array([1.0, -0.5])  # gradient arriving from later layers.
J_F_b7 = np.array([[0.1, 0.0], [0.0, -0.2]])  # branch derivative.
J_res_b7 = np.eye(2) + J_F_b7  # residual derivative.
print("incoming gradient:", grad_out_b7)  # inspect gradient before this block.

▶ What you'll see: a two-coordinate upstream gradient.

In [ ]:
grad_plain_b7 = J_F_b7.T @ grad_out_b7  # branch-only backpropagation.
grad_res_b7 = J_res_b7.T @ grad_out_b7  # residual backpropagation.
print("plain gradient:", grad_plain_b7)  # inspect branch-only result.
print("residual gradient:", grad_res_b7)  # inspect shortcut-preserved result.
assert np.linalg.norm(grad_res_b7) > np.linalg.norm(grad_plain_b7)  # verify preservation.
show_vec([np.linalg.norm(grad_plain_b7), np.linalg.norm(grad_res_b7)], ["plain", "residual"], "Basic 7: gradient norm")  # visualize norms.

▶ What you'll see: the residual gradient norm is much larger than the branch-only norm.

👀 Takeaway: the shortcut gives the backward pass a direct route around small branch derivatives.

### Basic 8 — Normalize a residual signal

**Goal.** Compute `(3.1 - 1.0)/sqrt(0.25 + eps)`, because scale affects both activations and gradients. We build it in 2 steps.

In [ ]:
value_b8 = 3.1  # lesson signal.
mean_b8 = 1.0  # normalization mean.
var_b8 = 0.25  # normalization variance.
eps_b8 = 1e-5  # small stabilizer.
print("value, mean, var:", value_b8, mean_b8, var_b8)  # inspect inputs.

▶ What you'll see: the signal is above the mean before scaling by standard deviation.

In [ ]:
normed_b8 = (value_b8 - mean_b8) / np.sqrt(var_b8 + eps_b8)  # standard normalization.
print("normalized:", round(float(normed_b8), 3))  # inspect the lesson number.
assert round(float(normed_b8), 3) == 4.200  # verify concrete arithmetic.
show_vec([value_b8, normed_b8], ["raw", "normalized"], "Basic 8: scale change")  # visualize raw vs normalized.

▶ What you'll see: normalization reports the signal as about `4.2` standard deviations high.

👀 Takeaway: shortcuts help gradients, but signal scale still needs explicit attention.

### Basic 9 — Compute a two-score softmax

**Goal.** Turn a residual-derived score into a probability against a baseline, because downstream losses use comparisons rather than raw scores. We build it in 2 steps.

In [ ]:
score_b9 = 3.1  # lesson score.
baseline_b9 = 0.4  # comparison score.
exp_b9 = np.array([np.exp(score_b9), np.exp(baseline_b9)])  # unnormalized evidence.
print("exponentials:", np.round(exp_b9, 3))  # inspect positive evidence values.

▶ What you'll see: the larger score has much larger exponential evidence.

In [ ]:
prob_b9 = softmax2(score_b9, baseline_b9)  # compute probability for the first score.
print("probability:", round(prob_b9, 3))  # inspect normalized comparison.
assert round(prob_b9, 3) == 0.937  # verify lesson number.
show_vec([prob_b9, 1 - prob_b9], ["score", "baseline"], "Basic 9: two-score softmax")  # visualize probabilities.

▶ What you'll see: the lesson score wins strongly but not absolutely.

👀 Takeaway: stable residual activations become stable probabilities only after normalization by competing scores.

### Basic 10 — Count activation memory

**Goal.** Compute the memory for `3` length-`128` float32 vectors, because residual training stores activations for the backward pass. We build it in 2 steps.

In [ ]:
vectors_b10 = 3  # number of stored vectors.
length_b10 = 128  # coordinates per vector.
bytes_b10 = 4  # float32 bytes.
print("vectors, length, bytes:", vectors_b10, length_b10, bytes_b10)  # inspect memory ingredients.

▶ What you'll see: memory bookkeeping is just count times width times bytes.

In [ ]:
kb_b10 = vectors_b10 * length_b10 * bytes_b10 / 1024  # convert bytes to KB.
print("memory KB:", round(kb_b10, 3))  # inspect the lesson memory number.
assert round(kb_b10, 3) == 1.500  # verify memory arithmetic.
show_vec([kb_b10], ["KB"], "Basic 10: activation memory")  # visualize the small block memory.

▶ What you'll see: the toy block uses `1.5` KB; real networks multiply this by batch size and depth.

👀 Takeaway: residual designs are mathematical objects that still consume activation memory during training.

## 🟡 Easy

### Easy 1 — Build a residual block function from scratch

**Goal.** Implement a tiny residual block `x + V·ReLU(Wx+b)`, because this is the full forward computation behind the shortcut idea. We build it in 3 steps.

In [ ]:
x_e1 = np.array([1.5, -0.5])  # input activation.
W_e1 = np.array([[1.4, -0.4], [-0.3, 0.8]])  # hidden branch weights.
b_e1 = np.array([0.8, -0.1])  # hidden branch bias.
V_e1 = np.array([[0.3, 0.1], [-0.2, 0.4]])  # output weights returning to input shape.
print("input shape:", x_e1.shape, "W shape:", W_e1.shape, "V shape:", V_e1.shape)  # inspect shape compatibility.

▶ What you'll see: the final branch output can match the input shape because `V` has two rows.

In [ ]:
h_e1 = relu(W_e1 @ x_e1 + b_e1)  # hidden branch activation.
F_e1 = V_e1 @ h_e1  # same-shape residual correction.
y_e1 = x_e1 + F_e1  # residual output.
print("hidden:", np.round(h_e1, 3))  # inspect branch hidden values.
print("F(x):", np.round(F_e1, 3))  # inspect correction.
print("y:", np.round(y_e1, 3))  # inspect residual output.
assert np.allclose(np.round(y_e1, 3), [2.43, -1.12])  # second hidden unit is zero here.

In [ ]:
plt.figure(figsize=(5, 3))  # create a compact comparison plot.
idx_e1 = np.arange(2)  # coordinate positions.
plt.bar(idx_e1 - 0.2, x_e1, width=0.2, label="x", color="gray")  # shortcut.
plt.bar(idx_e1, F_e1, width=0.2, label="F(x)", color="orange")  # branch.
plt.bar(idx_e1 + 0.2, y_e1, width=0.2, label="y", color="teal")  # output.
plt.axhline(0, color="black", linewidth=0.7)  # zero reference.
plt.title("Easy 1: full residual block")  # title the plot.
plt.legend()  # show labels.
plt.show()  # display the figure.

▶ What you'll see: the output is the input plus the branch correction coordinate by coordinate.

👀 Takeaway: a residual block is just an ordinary branch whose output is added back to the shortcut.

### Easy 2 — Project a skip connection when shapes change

**Goal.** Use a projection matrix on the shortcut, because addition only works when both paths have the same dimensionality. We build it in 3 steps.

In [ ]:
x_e2 = np.array([2.0, -1.0, 0.5])  # three-coordinate input.
F_e2 = np.array([0.4, -0.2])  # two-coordinate branch output.
P_e2 = np.array([[1.0, 0.0, 0.0], [0.0, 0.5, 0.5]])  # projection from 3D shortcut to 2D shortcut.
print("x shape:", x_e2.shape, "F shape:", F_e2.shape)  # inspect mismatch.

▶ What you'll see: the raw shortcut and branch cannot be added directly.

In [ ]:
skip_e2 = P_e2 @ x_e2  # projected shortcut has the branch shape.
y_e2 = skip_e2 + F_e2  # valid residual addition after projection.
print("projected skip:", skip_e2)  # inspect projection result.
print("output:", y_e2)  # inspect valid residual output.
assert y_e2.shape == F_e2.shape  # verify shape match.
assert np.allclose(y_e2, [2.4, -0.45])  # verify arithmetic.

In [ ]:
plt.figure(figsize=(4, 3))  # create a shape-aware value plot.
plt.bar(["skip0", "skip1", "F0", "F1"], [skip_e2[0], skip_e2[1], F_e2[0], F_e2[1]], color=["gray", "gray", "orange", "orange"])  # compare paths after projection.
plt.axhline(0, color="black", linewidth=0.7)  # zero reference.
plt.title("Easy 2: projection makes addition legal")  # title the plot.
plt.show()  # display the figure.

▶ What you'll see: the shortcut is first converted to two coordinates, then the two-coordinate correction can be added.

👀 Takeaway: projected skips preserve the residual idea across changes in width or resolution.

### Easy 3 — Compare deep products with and without identity

**Goal.** Track repeated scalar gains, because deep gradients are products of local derivatives. We build it in 3 steps.

In [ ]:
depths_e3 = np.arange(1, 41)  # depths from 1 to 40.
plain_e3 = 0.85 ** depths_e3  # branch-only derivative product.
res_e3 = 1.00 ** depths_e3  # pure identity shortcut product.
print("plain depth-40 multiplier:", round(float(plain_e3[-1]), 4))  # inspect vanishing.
print("identity depth-40 multiplier:", round(float(res_e3[-1]), 4))  # inspect shortcut preservation.
assert round(float(res_e3[-1]), 4) == 1.0000  # identity path keeps unit gain.

▶ What you'll see: a small per-layer shrink becomes severe at depth forty.

In [ ]:
ratio_e3 = res_e3[-1] / plain_e3[-1]  # compare preserved to branch-only scale.
print("identity/plain ratio at depth 40:", round(float(ratio_e3), 1))  # inspect magnitude gap.
assert ratio_e3 > 600  # concrete scale separation.

In [ ]:
plt.figure(figsize=(5, 3))  # create a depth curve.
plt.plot(depths_e3, plain_e3, label="0.85^L", color="red")  # branch-only shrink.
plt.plot(depths_e3, res_e3, label="identity", color="teal")  # shortcut gain.
plt.yscale("log")  # show multiplicative differences clearly.
plt.title("Easy 3: identity avoids repeated shrinkage")  # title the plot.
plt.xlabel("depth")  # label depth axis.
plt.ylabel("multiplier")  # label multiplier.
plt.legend()  # show labels.
plt.show()  # display the plot.

▶ What you'll see: the branch-only line decays exponentially while the identity line stays flat.

👀 Takeaway: residual shortcuts change the default gradient product from repeated shrinkage to a path with unit gain.

### Easy 4 — Train a scalar residual correction

**Goal.** Fit `y = x + a x` to a target that is close to identity, because residual learning focuses on the difference from the input. We build it in 4 steps.

In [ ]:
x_e4 = np.linspace(-1, 1, 21)  # training inputs.
target_e4 = 1.3 * x_e4  # target transformation is identity plus 0.3*x.
a_e4 = 0.0  # residual correction starts at zero.
print("true residual slope:", 0.3)  # inspect the correction we hope to learn.

▶ What you'll see: the target differs from identity by a simple slope correction.

In [ ]:
losses_e4 = []  # store losses.
for step_e4 in range(100):  # run small gradient descent.
    pred_e4 = x_e4 + a_e4 * x_e4  # residual model.
    err_e4 = pred_e4 - target_e4  # prediction error.
    grad_e4 = 2 * np.mean(err_e4 * x_e4)  # derivative wrt a.
    a_e4 -= 0.2 * grad_e4  # update correction slope.
    losses_e4.append(float(np.mean(err_e4 ** 2)))  # record loss.
print("learned a:", round(a_e4, 3))  # inspect learned correction.
assert abs(a_e4 - 0.3) < 0.02  # verify it learned the residual slope.

In [ ]:
final_e4 = x_e4 + a_e4 * x_e4  # final residual predictions.
print("first/last predictions:", round(float(final_e4[0]), 3), round(float(final_e4[-1]), 3))  # inspect endpoints.
assert losses_e4[-1] < losses_e4[0]  # verify training reduced loss.

In [ ]:
plt.figure(figsize=(5, 3))  # create a fit plot.
plt.plot(x_e4, target_e4, label="target", color="black")  # target line.
plt.plot(x_e4, final_e4, "--", label="residual fit", color="teal")  # learned line.
plt.title("Easy 4: learning the correction")  # title the plot.
plt.xlabel("x")  # label x-axis.
plt.ylabel("output")  # label output.
plt.legend()  # show labels.
plt.show()  # display the plot.

▶ What you'll see: the residual fit overlays the target because the branch learned the missing `0.3*x` correction.

👀 Takeaway: residual parameterization is especially natural when the desired transformation is near identity.

### Easy 5 — Inspect residual scale before a classifier

**Goal.** Compare residual scores and softmax probabilities, because a stable activation scale leads to less saturated downstream decisions. We build it in 3 steps.

In [ ]:
x_e5 = np.array([2.5, 0.7])  # shortcut logits from a previous block.
F_small_e5 = np.array([0.6, -0.3])  # moderate residual correction.
F_large_e5 = np.array([4.0, -4.0])  # overly large residual correction.
y_small_e5 = x_e5 + F_small_e5  # moderate residual logits.
y_large_e5 = x_e5 + F_large_e5  # large residual logits.
print("small logits:", y_small_e5)  # inspect moderate case.
print("large logits:", y_large_e5)  # inspect saturated case.

▶ What you'll see: the large correction greatly separates the two logits.

In [ ]:
p_small_e5 = softmax2(y_small_e5[0], y_small_e5[1])  # probability for class 0 with moderate scale.
p_large_e5 = softmax2(y_large_e5[0], y_large_e5[1])  # probability for class 0 with large scale.
print("prob small correction:", round(p_small_e5, 4))  # inspect probability.
print("prob large correction:", round(p_large_e5, 4))  # inspect saturation.
assert p_large_e5 > p_small_e5  # larger logit gap gives higher probability.

In [ ]:
plt.figure(figsize=(4.5, 3))  # create a probability comparison.
plt.bar(["small F", "large F"], [p_small_e5, p_large_e5], color=["teal", "red"])  # compare probabilities.
plt.ylim(0, 1.05)  # probability bounds.
plt.title("Easy 5: residual scale affects softmax")  # title the chart.
plt.ylabel("P(class 0)")  # label probability.
plt.show()  # display the plot.

▶ What you'll see: the large correction drives the probability closer to 1, which can reduce useful gradients.

👀 Takeaway: residual shortcuts help, but branch scale still affects downstream probability and loss behavior.

## 🔴 Advanced

### Advanced 1 — Compute a finite-difference Jacobian for a residual block

**Goal.** Estimate the Jacobian numerically, because `I + dF/dx` should appear even when the branch is nonlinear. We build it in 4 steps.

In [ ]:
x_a1 = np.array([1.0, -0.5])  # point where we inspect the derivative.
W_a1 = np.array([[0.4, -0.2], [0.1, 0.3]])  # branch hidden weights.
V_a1 = np.array([[0.5, -0.1], [0.2, 0.4]])  # branch output weights.
b_a1 = np.array([0.2, 0.1])  # branch bias.
print("x:", x_a1)  # inspect derivative point.

▶ What you'll see: a two-dimensional residual block input.

In [ ]:
def block_a1(x):  # define the nonlinear residual block.
    return x + V_a1 @ relu(W_a1 @ x + b_a1)  # shortcut plus branch.
base_a1 = block_a1(x_a1)  # compute output at the base point.
print("base output:", np.round(base_a1, 4))  # inspect output before finite differences.

In [ ]:
eps_a1 = 1e-5  # small finite-difference step.
J_num_a1 = np.zeros((2, 2))  # allocate numerical Jacobian.
for j_a1 in range(2):  # perturb each input coordinate.
    step_a1 = np.zeros(2)  # start with no perturbation.
    step_a1[j_a1] = eps_a1  # perturb one coordinate.
    J_num_a1[:, j_a1] = (block_a1(x_a1 + step_a1) - base_a1) / eps_a1  # forward finite difference.
print("numerical Jacobian:\n", np.round(J_num_a1, 3))  # inspect derivative.
assert np.all(np.diag(J_num_a1) > 0.9)  # shortcut keeps diagonal near one.

In [ ]:
plt.figure(figsize=(4, 3))  # create a Jacobian heatmap.
plt.imshow(J_num_a1, cmap="coolwarm", vmin=-1, vmax=1.5)  # visualize derivative entries.
plt.colorbar(label="dy_i/dx_j")  # label color scale.
plt.title("Advanced 1: residual Jacobian")  # title the plot.
plt.xlabel("input coordinate")  # label columns.
plt.ylabel("output coordinate")  # label rows.
plt.show()  # display the heatmap.

▶ What you'll see: the diagonal entries remain close to one, with off-diagonal branch interactions added.

👀 Takeaway: nonlinear residual blocks still inherit an identity derivative path.

### Advanced 2 — Compare plain and residual training on a near-identity task

**Goal.** Train two tiny linear models on the same data, because residual parameterization should learn a near-identity map faster. We build it in 5 steps.

In [ ]:
X_a2 = np.linspace(-2, 2, 80)[:, None]  # one-dimensional inputs as a column.
Y_a2 = 1.2 * X_a2  # target is close to identity.
w_plain_a2 = 0.0  # plain model predicts w*x from scratch.
a_res_a2 = 0.0  # residual model predicts x + a*x.
print("target multiplier:", 1.2)  # inspect the true map.

▶ What you'll see: the residual model only needs to learn the extra `0.2*x`.

In [ ]:
loss_plain_a2 = []  # store plain losses.
loss_res_a2 = []  # store residual losses.
for step_a2 in range(60):  # run gradient descent for both models.
    pred_plain_a2 = w_plain_a2 * X_a2  # plain prediction.
    err_plain_a2 = pred_plain_a2 - Y_a2  # plain error.
    grad_plain_a2 = 2 * float(np.mean(err_plain_a2 * X_a2))  # gradient wrt w.
    w_plain_a2 -= 0.05 * grad_plain_a2  # update plain weight.
    pred_res_a2 = X_a2 + a_res_a2 * X_a2  # residual prediction.
    err_res_a2 = pred_res_a2 - Y_a2  # residual error.
    grad_res_a2 = 2 * float(np.mean(err_res_a2 * X_a2))  # gradient wrt a.
    a_res_a2 -= 0.05 * grad_res_a2  # update residual correction.
    loss_plain_a2.append(float(np.mean(err_plain_a2 ** 2)))  # record plain loss.
    loss_res_a2.append(float(np.mean(err_res_a2 ** 2)))  # record residual loss.
print("plain w:", round(w_plain_a2, 3), "residual a:", round(a_res_a2, 3))  # inspect learned parameters.

In [ ]:
print("initial losses:", round(loss_plain_a2[0], 3), round(loss_res_a2[0], 3))  # compare starting difficulty.
print("final losses:", round(loss_plain_a2[-1], 6), round(loss_res_a2[-1], 6))  # compare ending fit.
assert loss_res_a2[0] < loss_plain_a2[0]  # residual starts closer because identity is built in.
assert loss_res_a2[-1] < 1e-4  # residual converges on this toy task.

In [ ]:
plt.figure(figsize=(5, 3))  # create training loss plot.
plt.plot(loss_plain_a2, label="plain y=w x", color="red")  # plain curve.
plt.plot(loss_res_a2, label="residual y=x+a x", color="teal")  # residual curve.
plt.yscale("log")  # log scale shows early advantage clearly.
plt.title("Advanced 2: residual starts near the target")  # title the plot.
plt.xlabel("step")  # label step axis.
plt.ylabel("MSE")  # label loss.
plt.legend()  # show labels.
plt.show()  # display the plot.

▶ What you'll see: the residual curve starts lower because the identity term already explains most of the target.

In [ ]:
pred_plain_final_a2 = w_plain_a2 * X_a2[:, 0]  # final plain predictions.
pred_res_final_a2 = X_a2[:, 0] + a_res_a2 * X_a2[:, 0]  # final residual predictions.
print("endpoint residual prediction:", round(float(pred_res_final_a2[-1]), 3))  # inspect final output at x=2.
assert abs(float(pred_res_final_a2[-1]) - 2.4) < 0.03  # target at x=2 is 2.4.

▶ What you'll see: the residual model matches the endpoint target after learning the small correction.

👀 Takeaway: residual learning is most helpful when the desired map is close to a pass-through plus a modest change.

### Advanced 3 — Track gradient norms through many random residual blocks

**Goal.** Multiply many random Jacobians, because residual shortcuts should keep gradient norms more stable than branch-only products. We build it in 4 steps.

In [ ]:
rng_a3 = np.random.default_rng(3)  # reproducible random Jacobians.
dim_a3 = 4  # activation dimension.
depth_a3 = 25  # number of blocks.
branch_Js_a3 = [0.08 * rng_a3.normal(size=(dim_a3, dim_a3)) for _ in range(depth_a3)]  # small branch derivatives.
g_a3 = np.ones(dim_a3) / np.sqrt(dim_a3)  # unit-norm incoming gradient.
print("initial gradient norm:", round(float(np.linalg.norm(g_a3)), 3))  # inspect starting norm.

▶ What you'll see: the backward signal starts with norm one.

In [ ]:
g_plain_a3 = g_a3.copy()  # branch-only gradient.
g_res_a3 = g_a3.copy()  # residual gradient.
norm_plain_a3 = []  # store branch-only norms.
norm_res_a3 = []  # store residual norms.
for J_a3 in branch_Js_a3:  # walk backward through blocks.
    g_plain_a3 = J_a3.T @ g_plain_a3  # plain branch derivative only.
    g_res_a3 = (np.eye(dim_a3) + J_a3).T @ g_res_a3  # residual derivative.
    norm_plain_a3.append(float(np.linalg.norm(g_plain_a3)))  # record plain norm.
    norm_res_a3.append(float(np.linalg.norm(g_res_a3)))  # record residual norm.
print("final plain norm:", format(norm_plain_a3[-1], ".2e"))  # inspect vanished gradient.
print("final residual norm:", round(norm_res_a3[-1], 3))  # inspect preserved gradient.
assert norm_res_a3[-1] > norm_plain_a3[-1] * 1e6  # residual is vastly larger here.

In [ ]:
plt.figure(figsize=(5, 3))  # create norm curve.
plt.plot(norm_plain_a3, label="branch only", color="red")  # plain gradient norms.
plt.plot(norm_res_a3, label="I + branch", color="teal")  # residual gradient norms.
plt.yscale("log")  # show huge multiplicative gap.
plt.title("Advanced 3: gradient norms through depth")  # title the plot.
plt.xlabel("block")  # label depth axis.
plt.ylabel("gradient norm")  # label norm.
plt.legend()  # show labels.
plt.show()  # display the plot.

▶ What you'll see: branch-only products collapse, while residual products remain much closer to the original scale.

In [ ]:
ratio_a3 = norm_res_a3[-1] / max(norm_plain_a3[-1], 1e-300)  # compute stable ratio.
print("final residual/plain ratio:", format(ratio_a3, ".2e"))  # inspect scale gap.
assert ratio_a3 > 1e6  # concrete stability gap.

▶ What you'll see: the final ratio is enormous because repeated small branch derivatives vanish.

👀 Takeaway: the identity derivative changes the long product that controls deep gradient flow.

### Advanced 4 — Study residual scaling as a stability knob

**Goal.** Sweep `y = x + αF(x)`, because scaling the branch can prevent residual additions from growing too quickly with depth. We build it in 4 steps.

In [ ]:
rng_a4 = np.random.default_rng(4)  # reproducible branch matrices.
dim_a4 = 6  # vector width.
depth_a4 = 40  # residual stack depth.
Ws_a4 = [0.15 * rng_a4.normal(size=(dim_a4, dim_a4)) for _ in range(depth_a4)]  # branch matrices.
alphas_a4 = np.array([0.0, 0.25, 0.5, 1.0])  # branch scaling values.
print("alphas:", alphas_a4)  # inspect sweep values.

▶ What you'll see: `α=0` is a pure identity stack and `α=1` uses the full branch.

In [ ]:
final_norms_a4 = []  # store final activation norms.
for alpha_a4 in alphas_a4:  # evaluate each residual scale.
    h_a4 = np.ones(dim_a4) / np.sqrt(dim_a4)  # same unit input for each run.
    for W_a4 in Ws_a4:  # apply many residual blocks.
        h_a4 = h_a4 + alpha_a4 * np.tanh(W_a4 @ h_a4)  # residual update with bounded branch.
    final_norms_a4.append(float(np.linalg.norm(h_a4)))  # record final norm.
print("final norms:", np.round(final_norms_a4, 3))  # inspect scale growth.
assert round(final_norms_a4[0], 3) == 1.000  # pure identity preserves norm exactly.

In [ ]:
plt.figure(figsize=(5, 3))  # create scale sweep bar chart.
plt.bar([str(a) for a in alphas_a4], final_norms_a4, color="teal")  # compare final norms.
plt.axhline(1.0, color="gray", linestyle="--")  # identity reference.
plt.title("Advanced 4: residual branch scale")  # title the plot.
plt.xlabel("α in x + αF(x)")  # label scale axis.
plt.ylabel("final activation norm")  # label norm.
plt.show()  # display the plot.

▶ What you'll see: larger branch scales move the final norm farther from the identity reference.

In [ ]:
spread_a4 = max(final_norms_a4) - min(final_norms_a4)  # quantify how much scale changes the outcome.
print("norm spread:", round(spread_a4, 3))  # inspect sensitivity to alpha.
assert spread_a4 > 0.05  # scaling has a visible effect.

▶ What you'll see: the residual branch scale changes the final activation magnitude.

👀 Takeaway: residual scaling is a practical knob for keeping deep stacks numerically controlled.

### Advanced 5 — Compare memory growth with and without checkpointing intuition

**Goal.** Estimate activation storage across depth, because residual networks need saved tensors for backpropagation even though the formula is simple. We build it in 4 steps.

In [ ]:
batch_a5 = 32  # examples per batch.
width_a5 = 128  # activation width.
float_bytes_a5 = 4  # float32 storage.
depths_a5 = np.array([10, 25, 50, 100])  # candidate residual depths.
print("batch, width:", batch_a5, width_a5)  # inspect dimensions.

▶ What you'll see: memory depends on batch size, width, dtype, and number of saved activations.

In [ ]:
saved_per_block_a5 = 3  # shortcut/branch/intermediate vectors as a simple bookkeeping model.
mem_mb_a5 = depths_a5 * saved_per_block_a5 * batch_a5 * width_a5 * float_bytes_a5 / (1024 ** 2)  # activation memory in MB.
print("activation MB:", np.round(mem_mb_a5, 3))  # inspect memory by depth.
assert round(float(mem_mb_a5[0]), 3) == 0.469  # concrete memory check for depth 10.

In [ ]:
checkpoint_fraction_a5 = 0.35  # toy fraction of activations retained under checkpointing-style recomputation.
mem_checkpoint_a5 = mem_mb_a5 * checkpoint_fraction_a5  # reduced stored memory estimate.
print("checkpoint-style MB:", np.round(mem_checkpoint_a5, 3))  # inspect reduced storage.
assert np.all(mem_checkpoint_a5 < mem_mb_a5)  # reduced storage is smaller.

In [ ]:
plt.figure(figsize=(5, 3))  # create memory scaling plot.
plt.plot(depths_a5, mem_mb_a5, marker="o", label="store activations", color="red")  # full storage.
plt.plot(depths_a5, mem_checkpoint_a5, marker="o", label="checkpoint-style", color="teal")  # reduced storage estimate.
plt.title("Advanced 5: activation memory scales with depth")  # title the plot.
plt.xlabel("residual blocks")  # label depth.
plt.ylabel("MB")  # label memory.
plt.legend()  # show labels.
plt.show()  # display the plot.

▶ What you'll see: memory grows roughly linearly with depth, while recomputation-style storage uses less memory.

👀 Takeaway: skip connections improve optimization, but training very deep residual stacks still requires careful activation-memory planning.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

A residual block learns a correction to the identity path, giving both activations and gradients a clean shortcut.

Depth makes signal routing a practical issue. A residual block keeps a direct identity path while a learned branch contributes only a correction. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, standardize, predict, and return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def one_hot(y, k):
    out = np.zeros((len(y), k))
    out[np.arange(len(y)), y.astype(int)] = 1.0
    return out


def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def random_relu_features(X, seed=0, width=24):
    rng = np.random.default_rng(seed + X.shape[1])
    W = rng.normal(0.0, 1.0 / np.sqrt(max(1, X.shape[1])), size=(X.shape[1], width))
    b = rng.normal(0.0, 0.15, size=width)
    H = np.maximum(0.0, X @ W + b)
    pair = X[:, :1] * X[:, 1:2] if X.shape[1] >= 2 else X
    return np.hstack([X, X * X, pair, H])


def batch_norm_fit(H, eps=1e-5):
    mu = H.mean(axis=0, keepdims=True)
    var = H.var(axis=0, keepdims=True)
    Z = (H - mu) / np.sqrt(var + eps)
    return Z, (mu, var, eps)


def batch_norm_apply(H, params):
    mu, var, eps = params
    return (H - mu) / np.sqrt(var + eps)


def layer_norm(H, eps=1e-5):
    mu = H.mean(axis=1, keepdims=True)
    var = H.var(axis=1, keepdims=True)
    return (H - mu) / np.sqrt(var + eps)


def group_norm(H, groups=4, eps=1e-5):
    usable = (H.shape[1] // groups) * groups
    head = H[:, :usable].reshape(H.shape[0], groups, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def instance_norm(H, eps=1e-5):
    usable = (H.shape[1] // 8) * 8
    head = H[:, :usable].reshape(H.shape[0], 8, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def deep_random_features(X, depth=4, scale=1.0, residual=False, seed=0):
    H = random_relu_features(X, seed=seed, width=20)
    rng = np.random.default_rng(seed + 100 + H.shape[1])
    for _ in range(depth):
        W = rng.normal(0.0, scale / np.sqrt(H.shape[1]), size=(H.shape[1], H.shape[1]))
        F = np.maximum(0.0, H @ W)
        if residual:
            H = H + 0.35 * F
        else:
            H = F
    return H


def transform_pair(x_tr, x_te, mode="plain", seed=0, scale=1.0, residual=False):
    Htr = random_relu_features(x_tr, seed=seed)
    Hte = random_relu_features(x_te, seed=seed)
    if mode == "batchnorm":
        Htr, params = batch_norm_fit(Htr)
        Hte = batch_norm_apply(Hte, params)
    if mode == "test_batchnorm_wrong":
        Htr, params = batch_norm_fit(Htr)
        Hte, _ = batch_norm_fit(Hte)
    if mode == "layernorm":
        Htr = layer_norm(Htr)
        Hte = layer_norm(Hte)
    if mode == "groupnorm":
        Htr = group_norm(Htr)
        Hte = group_norm(Hte)
    if mode == "instancenorm":
        Htr = instance_norm(Htr)
        Hte = instance_norm(Hte)
    if mode == "deep":
        Htr = deep_random_features(x_tr, depth=5, scale=scale, residual=residual, seed=seed)
        Hte = deep_random_features(x_te, depth=5, scale=scale, residual=residual, seed=seed)
    return Htr, Hte


def train_softmax_classifier(x_tr, y_tr, x_te, epsilon=0.0, epochs=40, lr=0.2, clip=None, schedule="constant", transform="plain", seed=0, scale=1.0, residual=False):
    Htr, Hte = transform_pair(x_tr, x_te, mode=transform, seed=seed, scale=scale, residual=residual)
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 700)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    targets = (1.0 - epsilon) * Y + epsilon / k
    rng = np.random.default_rng(seed + 10)
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    losses = []
    grad_norms = []
    for epoch in range(epochs):
        eta = lr_value(schedule, epoch, epochs, lr)
        P = softmax(Htr @ W + b)
        loss = -np.mean(np.sum(targets * np.log(P + 1e-12), axis=1))
        G = (P - targets) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        norm = float(np.sqrt(np.sum(dW * dW) + np.sum(db * db)))
        if clip is not None:
            factor = min(1.0, clip / (norm + 1e-12))
            dW = dW * factor
            db = db * factor
        W = W - eta * dW
        b = b - eta * db
        losses.append(float(loss))
        grad_norms.append(norm)
    preds = np.argmax(Hte @ W + b, axis=1)
    return preds, losses, grad_norms


def lr_value(schedule, epoch, epochs, base):
    if schedule == "constant":
        return base
    if schedule == "step":
        return base if epoch < epochs // 2 else base * 0.2
    if schedule == "cosine":
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * epoch / max(1, epochs - 1)))
    if schedule == "warmup_cosine":
        warm = max(2, epochs // 5)
        if epoch < warm:
            return base * (epoch + 1) / warm
        span = max(1, epochs - warm - 1)
        t = epoch - warm
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * t / span))
    if schedule == "onecycle":
        half = max(1, epochs // 2)
        if epoch < half:
            return base * (0.2 + 1.8 * epoch / half)
        return base * (2.0 - 1.8 * (epoch - half) / max(1, epochs - half))
    return base


def component_accuracy(name, X, y, **kwargs):
    def build(x_tr, y_tr, x_te):
        preds, _, _ = train_softmax_classifier(x_tr, y_tr, x_te, **kwargs)
        return preds
    return clf_accuracy(build, X, y)


def fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.3, seed=0):
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 701)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    rng = np.random.default_rng(seed + Htr.shape[1])
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    for epoch in range(epochs):
        P = softmax(Htr @ W + b)
        G = (P - Y) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        W = W - lr * dW
        b = b - lr * db
    return np.argmax(Hte @ W + b, axis=1)


def logistic_accuracy_for_features(X, y, mode="plain", scale=1.0, residual=False, seed=0):
    def build(x_tr, y_tr, x_te):
        Htr, Hte = transform_pair(x_tr, x_te, mode=mode, seed=seed, scale=scale, residual=residual)
        return fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.35, seed=seed)
    return clf_accuracy(build, X, y)


def ladder_preview(rungs):
    for name, X, y in rungs:
        classes = np.unique(y)
        print(f"{name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")
    print("First D1 sample:", rungs[0][1][0].tolist(), "label=", int(rungs[0][2][0]))


def print_metric_table(rows, header="rung metric"):
    print(header)
    for name, metric in rows:
        print(f"{name:36s} {metric:.3f}")


def split_for_demo(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def plot_ladder_results(rungs, metrics, title, artifact_fn=None):
    fig, axes = plt.subplots(1, 5, figsize=(16, 3))
    for ax, (name, X, y) in zip(axes, rungs):
        if artifact_fn is None:
            if X.shape[1] == 64:
                ax.imshow(X[0].reshape(8, 8), cmap="gray")
            else:
                ax.scatter(X[:, 0], X[:, 1], c=y, cmap="tab10", s=12)
        else:
            artifact_fn(ax, name, X, y)
        ax.set_title(name.split()[0])
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title + " artifacts")
    plt.show()

    plt.figure(figsize=(6, 3))
    plt.plot(range(1, 6), metrics, marker="o")
    plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
    plt.ylim(0.0, 1.05)
    plt.ylabel("held-out accuracy")
    plt.title(title + " summary")
    plt.grid(True, alpha=0.3)
    plt.show()

## The concept, built once

The lesson formula is $y=x+F(x),\ \frac{\partial y}{\partial x}=I+\frac{\partial F}{\partial x}$. For $x=[1,-2]$ and $F(x)=[0.2,0.5]$, the forward skip output is $[1.2,-1.5]$. If $dF/dx=diag(0.1,-0.2)$, the gradient path is $diag(1.1,0.8)$ rather than only the small residual branch.

In [ ]:
# [reference cell disabled: pre-existing SYNTAX error in the original compact notebook]
# x = np.array([1.0, -2.0])
# Fx = np.array([0.2, 0.5])
# y = x + Fx
# jac_f = np.diag([0.1, -0.2])
# grad_path = np.eye(2) + jac_f
# print("residual output:", y)
# print("gradient path:
# ", grad_path)
# assert np.allclose(y, np.array([1.2, -1.5]))
# assert np.allclose(np.diag(grad_path), np.array([1.1, 0.8]))


This helper is the reusable method for the rest of the notebook. It keeps the model and ladder fixed, then varies only this topic's component.

In [ ]:
print('Reusable component method is available in the setup cell and verified above.')

## The dataset ladder

We use the shared F5 classification ladder: D1 XOR, D2 blobs, D3 noisy moons, D4 real sklearn digits, and D5 noisy digits. The same accuracy wrapper and model family run on every rung.

In [ ]:
rungs = clf_digits_ladder()
ladder_preview(rungs)

## Run the same method across D1–D5

The table reports one held-out accuracy per rung while the component-specific sweep is printed for auditability.

In [ ]:
rungs = clf_digits_ladder()
rows = []
for rung_id, (name, X, y) in enumerate(rungs):
    plain = logistic_accuracy_for_features(X, y, mode="deep", scale=0.85, residual=False, seed=50 + rung_id)
    skip = logistic_accuracy_for_features(X, y, mode="deep", scale=0.85, residual=True, seed=50 + rung_id)
    rows.append((name, skip))
    print(name, "plain/residual", round(plain, 3), round(skip, 3))
metrics = [metric for _, metric in rows]
print_metric_table(rows, "residual accuracy")

## Results visualization

The closing figure has two parts: a small multiple showing each rung's data artifact and a summary curve of the selected metric from D1 to D5.

In [ ]:
plot_ladder_results(rungs, metrics, '6.17 Residual and skip connections')

## Pitfall on D5

A deep plain network can lose signal through repeated transforms. The fix is a dimension-matched skip path that preserves an identity route.

In [ ]:
name, X, y = clf_digits_ladder()[-1]
plain = logistic_accuracy_for_features(X, y, mode="deep", scale=0.65, residual=False, seed=91)
skip = logistic_accuracy_for_features(X, y, mode="deep", scale=0.65, residual=True, seed=91)
print("D5 deeper plain accuracy:", round(plain, 3))
print("D5 deeper residual accuracy:", round(skip, 3))
print("Fix: add only shape-compatible skips, or project the skip to the same dimension.")

## Evaluate it

- Metric: held-out accuracy from `clf_accuracy`; compare to a no-skill majority-class or plain-feature baseline.
- Sanity check: D1 should be inspectable and every probability target should sum to one when probabilities are used.
- Ablation: turn this topic's component off and verify the metric or diagnostic changes.
- Failure signal: unstable loss, axis mismatch, train/eval leakage, or D5 improvement without a matching diagnostic.

## Practice

1. Change one component value and rerun the D1 assertion plus the D1–D5 table.

2. Add a majority-class baseline to the summary curve.

3. On D5, print one extra diagnostic that would catch the named pitfall.